In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster


# ===============================
# 1. Load Dataset
# ===============================

df = pd.read_csv("UNSW_NB15_training-set.csv")

df_attack = df[df["attack_cat"] != "Normal"].copy()

# ===============================
# 2. Prepare Features
# ===============================

X = df_attack.drop(["label", "attack_cat"], axis=1)

categorical_cols = X.select_dtypes(include=["object"]).columns

for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ===============================
# 3. Compute Centroids per Attack Type
# ===============================

attack_categories = df_attack["attack_cat"].unique()

centroids = []
labels = []

for cat in attack_categories:
    class_samples = X_scaled[df_attack["attack_cat"] == cat]
    centroid = class_samples.mean(axis=0)
    centroids.append(centroid)
    labels.append(cat)

centroids = np.array(centroids)

# ===============================
# 4. Hierarchical Clustering
# ===============================

Z = linkage(centroids, method='average', metric='cosine')

# ===============================
# 5. Form 3 Clusters
# ===============================

cluster_labels = fcluster(Z, 3, criterion='maxclust')

grouping = pd.DataFrame({
    "Attack_Category": labels,
    "Cluster": cluster_labels
})

print(grouping.sort_values("Cluster"))

C:\Users\Kruthika\AppData\Local\Temp\ipykernel_19292\1738319924.py:24: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=["object"]).columns


  Attack_Category  Cluster
3       Shellcode        1
2         Fuzzers        1
5        Exploits        1
4  Reconnaissance        1
7           Worms        1
1        Analysis        2
0        Backdoor        2
6             DoS        2
8         Generic        3


In [1]:
# =====================================================
# Balance Clustered Dataset Using SMOTE
# =====================================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline


# =====================================================
# 1. Load Dataset
# =====================================================

df = pd.read_csv("UNSW_NB15_training-set.csv")

# Map cluster labels (your clustering result)

cluster_map = {
    "Shellcode": 1,
    "Fuzzers": 1,
    "Exploits": 1,
    "Reconnaissance": 1,
    "Worms": 1,
    "Analysis": 2,
    "Backdoor": 2,
    "DoS": 2,
    "Generic": 3
}

df["Cluster"] = df["attack_cat"].map(cluster_map)
df.loc[df["attack_cat"] == "Normal", "Cluster"] = 0

# Drop rows without cluster mapping (if any)
df = df.dropna(subset=["Cluster"])

y = df["Cluster"]
X = df.drop(["Cluster"], axis=1)

# Remove label + attack_cat if present
X = X.drop(["label", "attack_cat"], axis=1)


# =====================================================
# 2. Preprocessing
# =====================================================

categorical_cols = X.select_dtypes(include=["object"]).columns
numerical_cols = X.select_dtypes(exclude=["object"]).columns

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols)
    ]
)


# =====================================================
# 3. SMOTE Oversampling
# =====================================================

smote = SMOTE(sampling_strategy={2: 40000})

pipeline = ImbPipeline(steps=[
    ("preprocessing", preprocessor),
    ("smote", smote)
])

X_resampled, y_resampled = pipeline.fit_resample(X, y)


# =====================================================
# 4. Show New Distribution (Counts + Percentages)
# =====================================================

print("\nOriginal Distribution (Counts):")
print(y.value_counts().sort_index())

print("\nOriginal Distribution (Percentages):")
print((y.value_counts(normalize=True).sort_index() * 100).round(2))


print("\nBalanced Distribution (Counts):")
print(pd.Series(y_resampled).value_counts().sort_index())

print("\nBalanced Distribution (Percentages):")
print((pd.Series(y_resampled).value_counts(normalize=True).sort_index() * 100).round(2))

C:\Users\Kruthika\AppData\Local\Temp\ipykernel_19292\3498817814.py:53: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=["object"]).columns



Original Distribution (Counts):
Cluster
0.0    56000
1.0    63331
2.0    16010
3.0    40000
Name: count, dtype: int64

Original Distribution (Percentages):
Cluster
0.0    31.94
1.0    36.12
2.0     9.13
3.0    22.81
Name: proportion, dtype: float64

Balanced Distribution (Counts):
Cluster
0.0    56000
1.0    63331
2.0    40000
3.0    40000
Name: count, dtype: int64

Balanced Distribution (Percentages):
Cluster
0.0    28.09
1.0    31.77
2.0    20.07
3.0    20.07
Name: proportion, dtype: float64


In [7]:
# =====================================================
# UNSW-NB15 4-Class Classification (Cluster Based)
# =====================================================

import pandas as pd
import numpy as np
import time
import random

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


# =====================================================
# 1. Load Dataset
# =====================================================

train_df = pd.read_csv("UNSW_NB15_training-set.csv")
test_df = pd.read_csv("UNSW_NB15_testing-set.csv")

print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)


# =====================================================
# 2. Map Attack Categories to Clusters
# =====================================================

cluster_map = {
    "Shellcode": 1,
    "Fuzzers": 1,
    "Exploits": 1,
    "Reconnaissance": 1,
    "Worms": 1,
    "Analysis": 2,
    "Backdoor": 2,
    "DoS": 2,
    "Generic": 3
}

# Map train
train_df["Cluster"] = train_df["attack_cat"].map(cluster_map)
train_df.loc[train_df["attack_cat"] == "Normal", "Cluster"] = 0

# Map test
test_df["Cluster"] = test_df["attack_cat"].map(cluster_map)
test_df.loc[test_df["attack_cat"] == "Normal", "Cluster"] = 0


# =====================================================
# 3. Prepare Features & Labels
# =====================================================

y_train = train_df["Cluster"]
y_test = test_df["Cluster"]

X_train = train_df.drop(["label", "attack_cat", "Cluster"], axis=1)
X_test = test_df.drop(["label", "attack_cat", "Cluster"], axis=1)


# =====================================================
# 4. Preprocessing
# =====================================================

categorical_cols = X_train.select_dtypes(include=["object"]).columns
numerical_cols = X_train.select_dtypes(exclude=["object"]).columns

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols)
    ]
)


# =====================================================
# 5. Define Models (Multi-class)
# =====================================================

base_models = {
    "XGBoost": XGBClassifier(objective="multi:softprob", num_class=4, eval_metric="mlogloss"),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "LDA": LinearDiscriminantAnalysis(),
    "LightGBM": LGBMClassifier(objective="multiclass", num_class=4),
    "Logistic Regression": LogisticRegression(max_iter=2000)
}

results = {}


# =====================================================
# 6. Train + Calibrate Models
# =====================================================

for name, base_model in base_models.items():

    print(f"\nTraining + Calibrating {name}...")

    calibrated_model = CalibratedClassifierCV(
        base_model,
        method="sigmoid",
        cv=5
    )

    pipeline = Pipeline(steps=[
        ("preprocessing", preprocessor),
        ("classifier", calibrated_model)
    ])

    start_train = time.time()
    pipeline.fit(X_train, y_train)
    end_train = time.time()

    train_time = end_train - start_train

    results[name] = {
        "model": pipeline,
        "train_time": train_time
    }

    print(f"{name} Training + Calibration Time: {train_time:.4f} seconds")


# =====================================================
# 7. Random Detection from Test Set
# =====================================================

random_index = random.randint(0, len(X_test) - 1)
sample = X_test.iloc[[random_index]]
true_label = y_test.iloc[random_index]

cluster_names = {
    0: "Benign",
    1: "Cluster 1",
    2: "Cluster 2",
    3: "Cluster 3"
}

print("\n=======================================")
print("Random Test Record Index:", random_index)
print("Actual Label:", cluster_names[true_label])
print("=======================================")

print("\n--- Calibrated Detection Results ---")

for name, info in results.items():

    model = info["model"]

    start_detect = time.time()
    prediction = model.predict(sample)[0]
    probs = model.predict_proba(sample)[0]
    end_detect = time.time()

    detect_time = end_detect - start_detect

    print(f"\n{name}")
    print("Predicted:", cluster_names[prediction])
    
    print("Probabilities:")
    for i in range(4):
        print(f"  {cluster_names[i]}: {probs[i]:.6f}")
    
    print(f"Detection Time: {detect_time:.6f} seconds")
    print(f"Training + Calibration Time: {info['train_time']:.4f} seconds")

Training shape: (175341, 45)
Testing shape: (82332, 45)

Training + Calibrating XGBoost...


C:\Users\Kruthika\AppData\Local\Temp\ipykernel_19292\4096555391.py:74: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=["object"]).columns


XGBoost Training + Calibration Time: 31.2252 seconds

Training + Calibrating Random Forest...
Random Forest Training + Calibration Time: 132.3176 seconds

Training + Calibrating LDA...
LDA Training + Calibration Time: 16.0860 seconds

Training + Calibrating LightGBM...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.013789 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6457
[LightGBM] [Info] Number of data points in the train set: 140272, number of used features: 186
[LightGBM] [Info] Start training from score -1.141375
[LightGBM] [Info] Start training from score -1.018368
[LightGBM] [Info] Start training from score -2.393513
[LightGBM] [Info] Start training from score -1.477847


c:\Users\Kruthika\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.013860 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6453
[LightGBM] [Info] Number of data points in the train set: 140273, number of used features: 187
[LightGBM] [Info] Start training from score -1.141382
[LightGBM] [Info] Start training from score -1.018355
[LightGBM] [Info] Start training from score -2.393521
[LightGBM] [Info] Start training from score -1.477855


...

c:\Users\Kruthika\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008474 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6451
[LightGBM] [Info] Number of data points in the train set: 140273, number of used features: 186
[LightGBM] [Info] Start training from score -1.141382
[LightGBM] [Info] Start training from score -1.018355
[LightGBM] [Info] Start training from score -2.393521
[LightGBM] [Info] Start training from score -1.477855
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


c:\Users\Kruthika\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM Training + Calibration Time: 127.6357 seconds

Training + Calibrating Logistic Regression...
Logistic Regression Training + Calibration Time: 416.8212 seconds

Random Test Record Index: 77640
Actual Label: Benign

--- Calibrated Detection Results ---

XGBoost
Predicted: Cluster 1
Probabilities:
  Benign: 0.133678
  Cluster 1: 0.654522
  Cluster 2: 0.194597
  Cluster 3: 0.017203
Detection Time: 0.072628 seconds
Training + Calibration Time: 31.2252 seconds

Random Forest
Predicted: Cluster 1
Probabilities:
  Benign: 0.133245
  Cluster 1: 0.816644
  Cluster 2: 0.046335
  Cluster 3: 0.003775
Detection Time: 0.229138 seconds
Training + Calibration Time: 132.3176 seconds

LDA
Predicted: Cluster 1
Probabilities:
  Benign: 0.246042
  Cluster 1: 0.707136
  Cluster 2: 0.041087
  Cluster 3: 0.005736
Detection Time: 0.029006 seconds
Training + Calibration Time: 16.0860 seconds

LightGBM
Predicted: Cluster 1
Probabilities:
  Benign: 0.141565
  Cluster 1: 0.702695
  Cluster 2: 0.146258
  Cl

c:\Users\Kruthika\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Kruthika\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Kruthika\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Kruthika\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Kruthika\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\valida

In [1]:
# ====== 0. Force non-GUI backend ======
import matplotlib
matplotlib.use('Agg')

# ====== 1. Imports ======
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.inspection import permutation_importance
import os

from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis


# ====== 2. Load Original UNSW Dataset ======
train_df = pd.read_csv("UNSW_NB15_training-set.csv")
test_df  = pd.read_csv("UNSW_NB15_testing-set.csv")


# ====== 3. Create Cluster Mapping ======
cluster_map = {
    "Shellcode": 1,
    "Fuzzers": 1,
    "Exploits": 1,
    "Reconnaissance": 1,
    "Worms": 1,
    "Analysis": 2,
    "Backdoor": 2,
    "DoS": 2,
    "Generic": 3
}

train_df["Cluster"] = train_df["attack_cat"].map(cluster_map)
train_df.loc[train_df["attack_cat"] == "Normal", "Cluster"] = 0

test_df["Cluster"] = test_df["attack_cat"].map(cluster_map)
test_df.loc[test_df["attack_cat"] == "Normal", "Cluster"] = 0


# ====== 4. Features & Labels ======
X_train = train_df.drop(columns=["label", "attack_cat", "Cluster"])
y_train = train_df["Cluster"]

X_test  = test_df.drop(columns=["label", "attack_cat", "Cluster"])
y_test  = test_df["Cluster"]

# Identify numeric vs categorical columns
numeric_cols = X_train.select_dtypes(include=["int64","float64","bool"]).columns.tolist()
categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()


# ====== 5. Preprocessing ======
preprocess = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols)
])


# ====== 6. Models ======
models = {
    "XGBoost": XGBClassifier(
        objective="multi:softprob",
        num_class=4,
        eval_metric="mlogloss",
        n_estimators=300,
        learning_rate=0.05,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),
    "Logistic Regression": LogisticRegression(
        max_iter=500
    ),
    "LDA": LinearDiscriminantAnalysis()
}


# ====== 7. Directory ======
os.makedirs("feature_importance_plots_diff_grp", exist_ok=True)


# ====== 8. Feature Importance Function ======
def plot_feature_importance(feat_names, importance, title, filename):

    indices = np.argsort(importance)[::-1]
    sorted_feat_names = [feat_names[i] for i in indices]
    sorted_importance = importance[indices]

    plt.figure(figsize=(12, max(8, len(feat_names)*0.3)))
    plt.barh(range(len(sorted_feat_names)), sorted_importance, align='center')
    plt.yticks(range(len(sorted_feat_names)), sorted_feat_names)
    plt.gca().invert_yaxis()
    plt.title(title)
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()


# ====== 9. Train & Evaluate ======
for name, model in models.items():

    print("\n" + "="*25)
    print(f"Training: {name}")
    print("="*25)

    clf = Pipeline([
        ("preprocessor", preprocess),
        ("model", model)
    ])

    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, digits=3))

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))


    # ===== Feature Importance =====
    if name in ["XGBoost", "Random Forest", "Logistic Regression"]:

        feature_names = numeric_cols.copy()

        if categorical_cols:
            cat_features = clf.named_steps["preprocessor"]\
                .named_transformers_["cat"]\
                .get_feature_names_out(categorical_cols)
            feature_names.extend(cat_features)

        if name in ["XGBoost", "Random Forest"]:
            importance = clf.named_steps["model"].feature_importances_

        elif name == "Logistic Regression":
            importance = np.mean(np.abs(clf.named_steps["model"].coef_), axis=0)

        plot_feature_importance(
            feature_names,
            importance,
            title=f"{name} Feature Importance",
            filename=f"feature_importance_plots_diff_grp/{name}_feature_importance.png"
        )

        perm_importance = permutation_importance(
            clf, X_test, y_test, n_repeats=5, random_state=42
        )

        print("\nTop 10 Features by Permutation Importance:")
        top_idx = perm_importance.importances_mean.argsort()[::-1][:10]
        for i in top_idx:
            print(f"{feature_names[i]}: {perm_importance.importances_mean[i]:.4f}")

C:\Users\Kruthika\AppData\Local\Temp\ipykernel_21748\2205855650.py:56: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()



Training: XGBoost

Classification Report:
              precision    recall  f1-score   support

         0.0      0.490     0.683     0.571     37000
         1.0      0.308     0.404     0.349     21112
         2.0      0.241     0.139     0.176      5349
         3.0      1.000     0.002     0.003     18871

    accuracy                          0.420     82332
   macro avg      0.510     0.307     0.275     82332
weighted avg      0.544     0.420     0.358     82332


Confusion Matrix:
[[25257 10232  1511     0]
 [11779  8525   808     0]
 [ 2864  1741   744     0]
 [11603  7214    25    29]]

Top 10 Features by Permutation Importance:
response_body_len: 0.0197
sttl: 0.0114
dinpkt: 0.0096
ct_flw_http_mthd: 0.0094
dmean: 0.0087
ct_srv_src: 0.0084
proto_a/n: 0.0084
spkts: 0.0083
dwin: 0.0062
dloss: 0.0053

Training: Random Forest

Classification Report:
              precision    recall  f1-score   support

         0.0      0.727     0.664     0.694     37000
         1.0      0.4